#Import modules

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

#Data

In [5]:
data = pd.read_csv('drive/My Drive/Colab Notebooks/Metabolite_prediction_ML/Tissue_res/Fudan_data_processed/fudan_tpm_ensembl_id.csv', index_col = 0)
data.index.name = None
data

,ENSG00000034510,ENSG00000156508,ENSG00000227097,ENSG00000184009,ENSG00000140988,ENSG00000096384,ENSG00000161016,ENSG00000112306,ENSG00000229117,ENSG00000087086,...,ENSG00000223400,ENSG00000160180,ENSG00000160182,ENSG00000214326,ENSG00000229880,ENSG00000228930,ENSG00000275167,ENSG00000226115,ENSG00000233767,ENSG00000274847
FUSCCTNBC182,2447.155381,956.960040,185.220607,449.733950,398.575769,331.961043,485.891503,1032.302152,410.780763,1997.442072,...,0.0,0.629655,0.589982,0.0,0.0,0.0,0.0,0.353989,0.000000,0.453832
FUSCCTNBC203,3615.273039,1264.796635,212.927405,613.107390,410.668291,491.207544,371.948287,961.535813,440.800609,1747.999510,...,0.0,1.102217,0.000000,0.0,0.0,0.0,0.0,0.495729,0.000000,0.127110
FUSCCTNBC202,3142.598252,1128.969744,23.449189,488.009336,1362.397880,1019.435393,724.029280,924.140245,1062.730790,1050.809965,...,0.0,0.640911,2.802464,0.0,0.0,0.0,0.0,0.000000,0.000000,0.615926
FUSCCTNBC252,3077.853838,1747.266355,46.534260,1970.791018,1017.886626,1072.102967,1878.768767,3677.076850,1007.322009,836.004482,...,0.0,0.000000,0.651219,0.0,0.0,0.0,0.0,1.172194,0.000000,0.500938
FUSCCTNBC187,3355.012056,1385.507343,140.305852,639.268155,714.213497,488.831249,554.244668,1772.914800,524.592853,3068.234283,...,0.0,2.752266,0.573078,0.0,0.0,0.0,0.0,1.031540,0.000000,0.352663
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FUSCCTNBC478,2089.976932,1044.355445,153.284510,546.111531,507.092984,284.833353,392.680622,720.050386,640.417632,2146.178083,...,0.0,1.091030,0.000000,0.0,0.0,0.0,0.0,0.000000,1.226743,1.258198
FUSCCTNBC475,1880.619141,1095.808585,40.975269,996.981628,572.619756,511.969661,359.219520,899.688318,513.686189,781.535580,...,0.0,11.632138,5.069404,0.0,0.0,0.0,0.0,0.608328,0.000000,0.311963
FUSCCTNBC476,5419.466045,708.611839,236.248170,603.593763,595.921308,513.001821,373.973293,527.592540,655.816273,631.554989,...,0.0,81.840094,67.537764,0.0,0.0,0.0,0.0,0.000000,0.000000,0.865869
FUSCCTNBC479,2310.399136,1110.229721,14.343541,607.186315,654.144769,526.013277,623.449097,739.444346,808.675394,949.856546,...,0.0,659.927355,1272.202144,0.0,0.0,0.0,0.0,0.000000,0.000000,0.502338


In [6]:
#Assert that genes are in the correct order

with open("drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/BulkRNABert_data/common_gene_id.txt") as f:
    gene_list = [line.strip() for line in f]

print(len(gene_list))


data = data.reindex(columns = gene_list)
data = data.fillna(0)
data.head(3)

19062


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000284519,ENSG00000284532,ENSG00000284535,ENSG00000284543,ENSG00000284557,ENSG00000284564,ENSG00000284574,ENSG00000284587,ENSG00000284595,ENSG00000284596
FUSCCTNBC182,6.215518,0.098330,13.518447,23.405559,20.275218,35.318320,28.088914,42.892647,14.836902,18.022147,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FUSCCTNBC203,9.601603,0.275405,20.333653,35.523523,25.738050,34.976597,48.910587,39.972761,37.498924,26.678667,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FUSCCTNBC202,4.304712,0.133451,12.231194,42.410906,12.042749,13.395575,13.802557,19.543761,34.329743,38.354096,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#BulkRNABert

In [7]:
!pip install --upgrade git+https://github.com/huggingface/transformers.git --quiet
!pip install torch --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.9/501.9 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.0.0.dev0 which is incompatible.


In [8]:
from huggingface_hub import hf_hub_download
import numpy as np
import pandas as pd
from transformers import AutoConfig, AutoModel, AutoTokenizer
import torch

In [9]:
config = AutoConfig.from_pretrained(
    "InstaDeepAI/BulkRNABert",
    trust_remote_code=True,
)

config.embeddings_layers_to_save = (4,)

tokenizer = AutoTokenizer.from_pretrained("InstaDeepAI/BulkRNABert", trust_remote_code=True)
model = AutoModel.from_pretrained(
    "InstaDeepAI/BulkRNABert",
    config=config,
    trust_remote_code=True,
)

# Determine the device to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device) # Move the model to the selected device


gene_expression_array = data.to_numpy()
gene_expression_array = np.log10(1 + gene_expression_array)
# assert gene_expression_array.shape[1] == config.n_genes # This assertion might fail with your data, keep it commented for now


arrays = []
for i in range(gene_expression_array.shape[0]):
  #Tokenize
  # Process each row as a single sample
  gene_expression_ids = tokenizer.batch_encode_plus(gene_expression_array[i:i+1, :], return_tensors="pt")["input_ids"]

  # Move the input tensor to the same device as the model
  gene_expression_ids = gene_expression_ids.to(device)

  # Compute BulkRNABert's embeddings
  gene_expression_mean_embeddings = model(gene_expression_ids)["embeddings_4"].mean(axis=1)
  gene_expression_mean_embeddings = gene_expression_mean_embeddings.detach().cpu().numpy()
  arrays.append(gene_expression_mean_embeddings)
  print(f'Sample {i+1} done!')


res = np.concatenate(arrays, axis=0)
res.shape

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

bulkrnabert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/BulkRNABert:
- bulkrnabert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/InstaDeepAI/BulkRNABert:
- tokenizer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


special_tokens_map.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/24.0M [00:00<?, ?B/s]

Sample 1 done!
Sample 2 done!
Sample 3 done!
Sample 4 done!
Sample 5 done!
Sample 6 done!
Sample 7 done!
Sample 8 done!
Sample 9 done!
Sample 10 done!
Sample 11 done!
Sample 12 done!
Sample 13 done!
Sample 14 done!
Sample 15 done!
Sample 16 done!
Sample 17 done!
Sample 18 done!
Sample 19 done!
Sample 20 done!
Sample 21 done!
Sample 22 done!
Sample 23 done!
Sample 24 done!
Sample 25 done!
Sample 26 done!
Sample 27 done!
Sample 28 done!
Sample 29 done!
Sample 30 done!
Sample 31 done!
Sample 32 done!
Sample 33 done!
Sample 34 done!
Sample 35 done!
Sample 36 done!
Sample 37 done!
Sample 38 done!
Sample 39 done!
Sample 40 done!
Sample 41 done!
Sample 42 done!
Sample 43 done!
Sample 44 done!
Sample 45 done!
Sample 46 done!
Sample 47 done!
Sample 48 done!
Sample 49 done!
Sample 50 done!
Sample 51 done!
Sample 52 done!
Sample 53 done!
Sample 54 done!
Sample 55 done!
Sample 56 done!
Sample 57 done!
Sample 58 done!
Sample 59 done!
Sample 60 done!
Sample 61 done!
Sample 62 done!
Sample 63 done!
S

(318, 256)

In [10]:
res_df = pd.DataFrame(res, index = data.index)
res_df

,0,1,2,3,4,5,6,7,8,9,...,246,247,248,249,250,251,252,253,254,255
FUSCCTNBC182,1.128905,0.581498,0.500188,-1.247887,0.404327,1.156383,0.283921,0.808748,0.446825,-0.137029,...,0.107003,-0.049919,-0.030789,0.046934,-0.003684,1.515877,0.427733,0.430305,-0.220300,0.520542
FUSCCTNBC203,1.161954,0.609621,0.135103,-1.492413,0.526668,1.369344,0.361129,0.626221,0.543929,-0.422217,...,-0.052114,-0.203274,-0.313981,0.165715,-0.018724,1.587705,0.432416,0.484507,-0.381848,0.510365
FUSCCTNBC202,1.055236,0.508751,0.465815,-1.437997,0.370885,1.144881,0.320053,0.655491,0.418194,-0.251168,...,-0.149166,-0.065510,-0.178252,0.205631,0.099215,1.572515,0.380563,0.421553,-0.292916,0.610586
FUSCCTNBC252,0.899483,0.396040,0.498234,-1.427619,0.516742,1.199477,0.303826,0.602582,0.249424,-0.442779,...,-0.041004,-0.181116,-0.182803,0.185286,-0.034991,1.497385,0.311775,0.349059,-0.324299,0.726735
FUSCCTNBC187,1.104876,0.644343,0.355119,-1.243668,0.485025,1.279804,0.313562,0.766587,0.516566,-0.189318,...,0.144886,-0.139371,-0.111004,0.112083,0.016362,1.549944,0.384328,0.448924,-0.198983,0.628989
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FUSCCTNBC478,1.081285,0.554924,0.454344,-1.292133,0.511422,1.381838,0.432313,0.927881,0.509341,-0.362902,...,0.175993,-0.050715,0.055682,0.039031,0.136583,1.603841,0.370833,0.508560,-0.225716,0.492677
FUSCCTNBC475,0.999206,0.517205,0.330183,-1.325585,0.613828,1.350921,0.230515,0.610721,0.601880,-0.347583,...,-0.071987,-0.129857,-0.054259,0.169364,0.024330,1.537865,0.309049,0.549243,-0.224759,0.695609
FUSCCTNBC476,0.878588,0.361845,0.477997,-1.471190,0.656492,1.354724,0.349951,0.680656,0.558141,-0.341633,...,-0.226393,-0.038257,0.088487,0.131611,0.141410,1.526928,0.355278,0.471404,-0.208908,0.593406
FUSCCTNBC479,0.966972,0.447010,0.326598,-1.319838,0.515700,1.360988,0.343640,0.742484,0.611666,-0.484507,...,-0.146208,-0.010847,0.049558,0.179570,0.128298,1.619097,0.403451,0.574757,-0.247531,0.636322


In [11]:
res_df.to_csv('drive/My Drive/Colab Notebooks/Deep_learning_metabolomics_prediction/BulkRNABert_data/fudan_bulkrnabert_embeddings.csv', index = True)

In [12]:
# ############# Original Code ##############

# config = AutoConfig.from_pretrained(
#     "InstaDeepAI/BulkRNABert",
#     trust_remote_code=True,
# )

# config.embeddings_layers_to_save = (4,)

# tokenizer = AutoTokenizer.from_pretrained("InstaDeepAI/BulkRNABert", trust_remote_code=True)
# model = AutoModel.from_pretrained(
#     "InstaDeepAI/BulkRNABert",
#     config=config,
#     trust_remote_code=True,
# )

# # Load bulk RNA-seq data and preprocess them.
# csv_path = hf_hub_download(
#     repo_id="InstaDeepAI/BulkRNABert",
#     filename="data/tcga_sample.csv",
#     repo_type="model",
# )
# gene_expression_array = pd.read_csv(csv_path).drop(["identifier"], axis=1).to_numpy()[:1, :]
# gene_expression_array = np.log10(1 + gene_expression_array)
# assert gene_expression_array.shape[1] == config.n_genes

# # Tokenize
# gene_expression_ids = tokenizer.batch_encode_plus(gene_expression_array, return_tensors="pt")["input_ids"]

# # Compute BulkRNABert's embeddings
# gene_expression_mean_embeddings = model(gene_expression_ids)["embeddings_4"].mean(axis=1)  # embeddings can be used for downstream tasks.